In [1]:
!py --version

Python 3.10.8


# deleted the cache 

In [1]:
# from huggingface_hub import scan_cache_dir
# scan_cache_dir().delete_revisions()

In [2]:
from transformers import LayoutLMv3Processor, LayoutLMv3ForTokenClassification
import torch 
from paddleocr import PaddleOCR
import paddle
import pprint
import numpy as np
from PIL import Image

D:\Anuj_S_Jain\Data\Code\Project\Society_MCE_QT\Society_MCE\Prototype\.venv\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
No model hoster is available! Please check your network connection to one of the following model hoster: HuggingFace (https://huggingface.co), ModelScope (https://modelscope.cn), AIStudio (https://aistudio.baidu.com), or BOS (https://paddle-model-ecology.bj.bcebos.com). Otherwise, only local models can be used.
D:\Anuj_S_Jain\Data\Code\Project\Society_MCE_QT\Society_MCE\Prototype\.venv\lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from

In [3]:
gpu_or_cpu="cuda" if torch.cuda.is_available() else "cpu"
device = torch.device(gpu_or_cpu) #used to set the model to run in cpu or cuda 

print("Using device:", device)

model = LayoutLMv3ForTokenClassification.from_pretrained("microsoft/layoutlmv3-base")
processor = LayoutLMv3Processor.from_pretrained("microsoft/layoutlmv3-base",apply_ocr=False)

# Move model to GPU
model.to(device)

Using device: cuda


Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

LayoutLMv3ForTokenClassification LOAD REPORT from: microsoft/layoutlmv3-base
Key                                | Status     | 
-----------------------------------+------------+-
layoutlmv3.embeddings.position_ids | UNEXPECTED | 
classifier.weight                  | MISSING    | 
classifier.bias                    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


OSError: Can't load processor for 'microsoft/layoutlmv3-base'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'microsoft/layoutlmv3-base' is the correct path to a directory containing a processor_config.json file

In [ ]:
image_path = "./training data/1.jpg"

In [ ]:
print(paddle.device.get_device())
ocr = PaddleOCR(use_textline_orientation=True, lang='en', device='gpu')
result = ocr.predict(image_path)

In [ ]:
# pprint.pprint(result)

In [ ]:
# result

In [ ]:
# pprint.pprint(len(result[0]["rec_texts"]))
# pprint.pprint(result[0]["rec_texts"])

In [ ]:
# pprint.pprint(len(result[0]["rec_polys"]))
# pprint.pprint(result[0]["rec_polys"])

In [ ]:
# pprint.pprint(result[0]["rec_texts"][0])
# pprint.pprint(result[0]["rec_polys"][0])

# print(result[0]["rec_polys"][0][:])
poly=result[0]["rec_polys"][0][:]

In [ ]:

# to get the max and min points 
def getMinMaxPoint(poly):
    x0 = int(poly[:,0].min())
    y0 = int(poly[:,1].min())
    x1 = int(poly[:,0].max())
    y1 = int(poly[:,1].max())
    return [x0, y0, x1, y1] 

# print(getMinMaxPoint(poly))
 # the rectangle for the **entire line**


In [ ]:
pprint.pprint(result[0]["rec_texts"][0].split())
# a=result[0]["rec_texts"][0].split()

# print(len(a[0].strip()))

# print(result[0]["rec_texts"][0])
# print(len(result[0]["rec_texts"][0]))
# print(len(a[0]),len(a[1]))

In [ ]:
token_word=[]
token_poly=[]

In [ ]:
# xy_points=getMinMaxPoint(poly)
# sentence= result[0]["rec_texts"][0]
# sentence_split = result[0]["rec_texts"][0].split()

def tokenize_sent(xy_points, sentence_split, sentence):
    print(len(sentence))
    letter_width=(xy_points[2]-xy_points[0])/len(sentence)
    x = xy_points[0]
    y0, y1 = xy_points[1], xy_points[3]

    for i, word in enumerate(sentence_split):
        word_width = len(word) * letter_width
        x_end = x + word_width

        # print([int(x), y0, int(x_end), y1])
        token_word.append(word)
        token_poly.append([int(x), y0, int(x_end), y1])
        x = x_end

        # add space after word (except last)
        if i < len(sentence_split) - 1:
            x += letter_width


print(tokenize_sent(xy_points,sentence_split,sentence))

In [ ]:
# token_word
# token_poly

In [ ]:
for i,(box,word) in enumerate(zip(result[0]["rec_polys"],result[0]["rec_texts"])):
    if len(word) == 0:
        continue  
    points_max_min=getMinMaxPoint(box[:])
    tokenize_sent(points_max_min,word.split(),word)
    print('index: ',i,word)
    # print(box[:],word)

In [ ]:
len(token_word)
# token_word

In [ ]:
len(token_poly)
# token_poly

In [ ]:
image = Image.open(image_path).convert("RGB")
width, height = image.size
print(width,height)

In [ ]:
boxes=[]
def normalize(token_ploy, width, height):
    for box in token_ploy:
        boxes.append([
            int(1000 * box[0] / width),
            int(1000 * box[1] / height),
            int(1000 * box[2] / width),
            int(1000 * box[3] / height),
            ])
normalize(token_poly,width,height)

In [ ]:
pprint.pprint(boxes)

In [ ]:
image = Image.open(image_path).convert("RGB")

In [ ]:
encoding = processor(
    images=image,
    text=token_word,
    boxes=boxes,
    return_tensors="pt",
    padding="max_length",
    truncation=True
)

In [ ]:
# Move tensors to the same device as model
for k, v in encoding.items():
    encoding[k] = v.to(device)

In [ ]:
# Get predictions
outputs = model(**encoding)
# print(outputs)
# The model outputs logits for each token
logits = outputs.logits  # shape: [batch_size, seq_len, num_labels]
print(logits)
# Get predicted class for each token
predictions = torch.argmax(logits, dim=-1)
# print(predictions)

In [ ]:
# Suppose your model has these label mappings
labels = model.config.id2label

predicted_labels = [labels[p.item()] for p in predictions[0][:len(token_word)]]

for token_word, label in zip(token_word, predicted_labels):
    print(token_word, "->", label)